# NINAAD WAGLE | BTECH AI SEM V | I065 B2 | NLP LAB 6

# Task a : Selecting a problem statement where NER is applicable

**The problem.** A news desk receives thousands of news sentences every day. Someone has
to read them to answer very ordinary questions - which people are in the news, which
organisations are being talked about, which places the reports come from, and which dates
are mentioned. Done by hand this does not scale, and the news cannot be searched by *who*
or *where* because that information is buried inside plain sentences.

**Why NER fits.** Those four questions are exactly the four kinds of thing a named entity
recogniser pulls out of text - person, organisation, place and time. If every sentence is
passed through NER, free text turns into a small table of who, where and when, and the
news can then be searched and counted instead of read.

**What is built here.** A tagger is learnt from a labelled news corpus, its tags are joined
back into whole entities, and those entities become a searchable index of news sentences
the tagger has never seen before.

# Task b : Identifying the dataset

The dataset is the **Annotated Corpus for Named Entity Recognition** (`NER dataset.csv`),
taken from the Groningen Meaning Bank. It is news text, which is the same kind of text the
problem statement is about, and every word in it has already been labelled by hand, so the
tagger has something to learn from and be checked against.

One row is one word, not one sentence. `Sentence #` is written only on the first word of a
sentence and left blank after that, so it is filled downwards to give every row its
sentence number. Ten rows have an empty word and are dropped.

In [1]:
import os
import pandas as pd

path = "/kaggle/input/entity-annotated-corpus/ner_dataset.csv"
if not os.path.exists(path):
    path = "NER dataset.csv"

data = pd.read_csv(path, encoding="latin1")
data["Sentence #"] = data["Sentence #"].ffill()
data = data.dropna(subset=["Word"])

print("words:", len(data), "| sentences:", data["Sentence #"].nunique())
data.head(10)

words: 1048565 | sentences: 47959


,Sentence #,Word,POS,Tag
0,Sentence: 1,Thousands,NNS,O
1,Sentence: 1,of,IN,O
2,Sentence: 1,demonstrators,NNS,O
3,Sentence: 1,have,VBP,O
4,Sentence: 1,marched,VBN,O
5,Sentence: 1,through,IN,O
6,Sentence: 1,London,NNP,B-geo
7,Sentence: 1,to,TO,O
8,Sentence: 1,protest,VB,O
9,Sentence: 1,the,DT,O


`Word` is the word itself, `POS` is its part of speech and `Tag` is the entity label.

The entity labels use the **IOB** scheme. `O` means the word is not part of any entity,
`B-x` means the word **begins** an entity of type `x`, and `I-x` means the word is
**inside** the entity that was just begun. So *Hyde Park* is tagged `B-geo I-geo` - two
words, one place.

In [2]:
meanings = {"geo": "place", "org": "organisation", "per": "person", "gpe": "country or nationality",
            "tim": "date or time", "art": "artifact", "eve": "event", "nat": "natural phenomenon"}

tags = data["Tag"].value_counts().rename_axis("tag").reset_index(name="count")
tags["meaning"] = tags["tag"].str[2:].map(meanings).fillna("not an entity")
tags

,tag,count,meaning
0,O,887898,not an entity
1,B-geo,37644,place
2,B-tim,20333,date or time
3,B-org,20143,organisation
4,I-per,17251,person
5,B-per,16990,person
6,I-org,16784,organisation
7,B-gpe,15870,country or nationality
8,I-geo,7414,place
9,I-tim,6528,date or time


`O` takes up about 85% of the rows, which is normal - most words in a sentence are not
names. Among the real entities, places, dates, organisations and people are common, while
`art`, `eve` and `nat` appear only a few hundred times in the whole corpus.

# Task c : Applying NER on the dataset

## i : Grouping the words back into sentences

An entity tag depends on context - *May* is a date in one sentence and a person in another -
so the words have to be read sentence by sentence and not as a flat list. Grouping the table
on the sentence number gives a list of sentences, where each sentence is a list of
(word, POS tag, entity tag) triples.

In [3]:
sentences = [list(zip(g["Word"], g["POS"], g["Tag"]))
             for _, g in data.groupby("Sentence #", sort=False)]

print("sentences:", len(sentences))
print(" ".join(word for word, pos, tag in sentences[0]))
sentences[0][:8]

sentences: 47959
Thousands of demonstrators have marched through London to protest the war in Iraq and demand the withdrawal of British troops from that country .


[('Thousands', 'NNS', 'O'),
 ('of', 'IN', 'O'),
 ('demonstrators', 'NNS', 'O'),
 ('have', 'VBP', 'O'),
 ('marched', 'VBN', 'O'),
 ('through', 'IN', 'O'),
 ('London', 'NNP', 'B-geo'),
 ('to', 'TO', 'O')]

## ii : A tagger that learns by counting

The first 80% of the sentences are used for learning and the last 20% are held back, so the
tagger is always judged on sentences it has never seen.

The learning itself is plain counting. For every word, count which entity tag it was given
across the training sentences and keep the most common one. *London* is tagged `B-geo`
nearly every time it appears, so `B-geo` is what the tagger will answer for it. A word that
never appeared in training has nothing to count, so a simple rule takes over - a capitalised
word is guessed to be the start of a person's name, anything else is `O`.

In [4]:
from collections import Counter, defaultdict

split = int(0.8 * len(sentences))
train, test = sentences[:split], sentences[split:]

tag_counts = defaultdict(Counter)
for sentence in train:
    for word, pos, tag in sentence:
        tag_counts[word][tag] += 1

lookup = {word: counts.most_common(1)[0][0] for word, counts in tag_counts.items()}

def predict(word):
    if word in lookup:
        return lookup[word]
    return "B-per" if word.istitle() else "O"

print("train sentences:", len(train), "| test sentences:", len(test), "| words learnt:", len(lookup))
print({word: lookup[word] for word in ["London", "Taleban", "Monday", "the"]})

train sentences: 38367 | test sentences: 9592 | words learnt: 31814
{'London': 'B-geo', 'Taleban': 'B-org', 'Monday': 'B-tim', 'the': 'O'}


## iii : Checking the tagger on unseen sentences

Accuracy on its own would be misleading, because 85% of the words are `O` - a tagger that
said `O` for everything would already look 85% correct. So the report below leaves `O` out
and scores only the entity tags.

**Precision** is how often the tagger was right when it claimed an entity, **recall** is how
many of the real entities it managed to find, and **f1-score** combines the two.

In [5]:
from sklearn.metrics import classification_report

predicted = [[predict(word) for word, pos, tag in sentence] for sentence in test]

y_true = [tag for sentence in test for word, pos, tag in sentence]
y_pred = [tag for sentence in predicted for tag in sentence]

print("token accuracy:", round(sum(a == b for a, b in zip(y_true, y_pred)) / len(y_true), 4))
print(classification_report(y_true, y_pred, labels=sorted(set(y_true) - {"O"}), digits=2, zero_division=0))

token accuracy: 0.9471


              precision    recall  f1-score   support

       B-art       0.07      0.02      0.04        82
       B-eve       0.61      0.30      0.41        46
       B-geo       0.79      0.85      0.82      7552
       B-gpe       0.95      0.95      0.95      3243
       B-nat       0.46      0.46      0.46        48
       B-org       0.67      0.50      0.57      4082
       B-per       0.52      0.74      0.61      3320
       B-tim       0.87      0.76      0.81      4105
       I-art       0.05      0.02      0.03        43
       I-eve       0.23      0.07      0.11        44
       I-geo       0.73      0.62      0.67      1408
       I-gpe       0.72      0.65      0.68        40
       I-nat       0.00      0.00      0.00        12
       I-org       0.72      0.54      0.61      3470
       I-per       0.76      0.66      0.71      3331
       I-tim       0.61      0.12      0.21      1308

   micro avg       0.75      0.70      0.72     32134
   macro avg       0.55   

The tagger gets about **95% of the words right**, and a weighted f1 of **0.71** over the
entity tags on their own.

It does well on the types that repeat. `B-gpe` scores 0.95 because words like *Iranian* and
*British* are nationalities every single time they appear. `B-geo` and `B-tim` are around
0.8 for the same reason - country names and weekdays are a short fixed list that the
training sentences cover. `B-org` and `B-per` are weaker at 0.57 and 0.61, because new
organisations and new people keep turning up and a word that was never seen cannot be
counted.

The rare types fail almost completely. `art`, `eve` and `nat` have only a few hundred
examples in the whole corpus, so there is nothing to count and `I-nat` scores 0. `I-tim` is
poor at 0.21 for a different reason - the second word of a date (*last year*, *three
weeks*) is usually a common word that is `O` most of the time, and counting always picks
the majority.

This is the weakness of a lookup tagger in one line: it knows the words it has seen and
guesses at the rest. It never looks at the surrounding words, which is what a CRF or a
neural tagger would add.

## iv : Joining B- and I- tags into whole entities

The tags are still one per word, so *United Nations* is two rows and not one organisation.
Chunking fixes that: a `B-` tag opens a new entity, an `I-` tag of the same type carries it
on, and an `O` closes whatever was open. The function below walks through a sentence once
and returns the finished entities with their type.

In [6]:
def entities(words, tags):
    found, part, label = [], [], None
    for word, tag in zip(words, tags):
        if tag.startswith("I-") and tag[2:] == label:
            part.append(word)
        else:
            if part:
                found.append((" ".join(part), label))
            part, label = ([word], tag[2:]) if tag != "O" else ([], None)
    if part:
        found.append((" ".join(part), label))
    return found

words = [word for word, pos, tag in test[104]]
print(" ".join(words))
print("real      :", entities(words, [tag for word, pos, tag in test[104]]))
print("predicted :", entities(words, predicted[104]))

North Korea 's official KCNA news agency says the government submitted its claim for damages to South Korea 's Human Rights Commission Friday .
real      : [('North Korea', 'geo'), ('KCNA', 'org'), ('South Korea', 'geo'), ('Human Rights Commission', 'org'), ('Friday', 'tim')]
predicted : [('North Korea', 'geo'), ('KCNA', 'org'), ('South Korea', 'geo'), ('Human Rights Commission', 'org'), ('Friday', 'tim')]


# Task d : Obtaining the solution - a searchable who / where / when index

Every held back sentence is now tagged and chunked, and its entities are written into four
columns - people (`per`), organisations (`org`), places (`geo`) and dates (`tim`). This
table is the answer to the problem statement: news sentences the tagger had never read,
turned into structured rows that a news desk can work with.

In [7]:
columns = {"per": "people", "org": "organisations", "geo": "places", "tim": "dates"}

rows = []
for sentence, tags in zip(test, predicted):
    words = [word for word, pos, tag in sentence]
    found = entities(words, tags)
    row = {"sentence": " ".join(words)}
    row.update({name: ", ".join(text for text, label in found if label == short)
                for short, name in columns.items()})
    rows.append(row)

news = pd.DataFrame(rows)
print("sentences indexed:", len(news))
news.head()

sentences indexed: 9592


,sentence,people,organisations,places,dates
0,In a letter to Egyptian President Hosni Mubara...,President Hosni Mubarak,Human Rights Watch,"New, Cairo",
1,The U.S. State Department and the European par...,,"State Department, European",U.S.,
2,Pakistani military officials say 14 of about 4...,,,Afghanistan,
3,Officials say the Frontier Corps paramilitary ...,,"Corps, Taliban","Frontier, Mohmand",
4,Military spokesman Major General Athar Abbas t...,Athar Abbas,General,"Jalalabad, Pakistan",Thursday


Counting each column gives the news desk its summary - who and what these sentences are
mostly about, without anybody reading them.

In [8]:
top = {name: Counter(entity for cell in news[name] for entity in cell.split(", ") if entity).most_common(5)
       for name in columns.values()}

pd.DataFrame({name: [f"{entity} ({count})" for entity, count in items] for name, items in top.items()})

,people,organisations,places,dates
0,Prime (145),U.N. (134),U.S. (874),Tuesday (297)
1,President Bush (96),NATO (107),Iraq (398),Thursday (271)
2,Mr. (96),Nations (100),Iran (319),Sunday (259)
3,President (75),Taleban (100),United States (273),Friday (249)
4,Mr. Bush (68),EU (88),Afghanistan (200),Monday (241)


And because the entities sit in columns, a sentence can now be looked up by the thing it
mentions instead of by the words it happens to use. This is the search a plain keyword
search cannot do - it matches on *Iran the country* rather than on the letters "iran".

In [9]:
mentions = news["people"] + " | " + news["organisations"] + " | " + news["places"] + " | " + news["dates"]

def search(name):
    return news[mentions.str.contains(name, regex=False)]

hits = search("Iran")
print("sentences mentioning Iran:", len(hits))
hits[["sentence", "people", "places", "dates"]].head()

sentences mentioning Iran: 302


,sentence,people,places,dates
28,Venezuela 's president has called on the inter...,,"Venezuela, Iran",
29,"Speaking on a trip to London Monday , Hugo Cha...",Hugo Chavez,"London, Europe, Iran",Monday
30,"Earlier , Mr. Chavez warned that the price of ...",Mr. Chavez,"United States, Iran",
47,The two governments have been locked in a thre...,,"U.S., Iran",three-year
49,Mr. Khatami 's remarks follow a published repo...,Mr. Khatami,"United States, Iran",Sunday


## What the solution gives, and where it falls short

The problem statement asked for a way to see who, where and when without reading the news.
That is now a two line job. `news` holds 9,592 unseen sentences as structured rows, the
counts say straight away that these sentences are mostly about the *U.S.*, *Iraq* and
*Iran* and mostly datelined *Tuesday*, and `search` pulls back every sentence that mentions
a given entity - matching on the entity itself, not on a keyword.

The mistakes from Task c follow through into the index. *Prime* is listed as a person
because *Prime Minister* gets split in half, titles like *President* are pulled into the
name next to them, and a few multi word places lose their second word when the `I-` tag is
missed. A counting tagger is a fair first solution because it is quick, needs no training
and is easy to explain, but cleaning up these errors needs a tagger that reads the words
around each word instead of each word on its own.